In [ ]:
%matplotlib inline
import sys, os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, '/Users/yotameviatar/vs_code')

from HR import hr_config as config
from HR import hr_processor as processor

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.35, 'grid.linestyle': '--'})
print('Setup complete')

In [ ]:
# ── Configuration overrides ───────────────────────────────────────────────────
SUBJECT_FILTER       = []
ANAL_TMIN            = -5.0
ANAL_TMAX            =  10.0
BASELINE_TMIN        = -5.0
BASELINE_TMAX        =  0.0
BASELINE_METHOD      = 'mean'
SCORE_TMIN           =  0.0
SCORE_TMAX           =  6.0
HR_MIN_BPM           =  30
HR_MAX_BPM           =  200
HR_Z_SCORE_THRESHOLD =  3.0

In [ ]:
# ── Load cache & apply config overrides ──────────────────────────────────────
CACHE_FILE = os.path.join(config.OUTPUT_DIR, '_cache', 'hr_cache.pkl')
with open(CACHE_FILE, 'rb') as f:
    _cache = pickle.load(f)

_all_trials = _cache.get('trials', _cache)

trials_data = {s: v for s, v in _all_trials.items()
               if not SUBJECT_FILTER or s in SUBJECT_FILTER}

config.ANAL_TMIN            = ANAL_TMIN
config.ANAL_TMAX            = ANAL_TMAX
config.BASELINE_TMIN        = BASELINE_TMIN
config.BASELINE_TMAX        = BASELINE_TMAX
config.BASELINE_METHOD      = BASELINE_METHOD
config.SCORE_TMIN           = SCORE_TMIN
config.SCORE_TMAX           = SCORE_TMAX
config.HR_MIN_BPM           = HR_MIN_BPM
config.HR_MAX_BPM           = HR_MAX_BPM
config.HR_Z_SCORE_THRESHOLD = HR_Z_SCORE_THRESHOLD

processor.apply_analysis_params(trials_data, config)

print(f'Loaded {len(trials_data)} subjects')
for subj, sessions in sorted(trials_data.items()):
    for sk, trials in sessions.items():
        n_rej = sum(t['rejected'] for t in trials)
        print(f'  {subj} {sk}: {len(trials)} trials, {n_rej} rejected')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def _pct(epoch, baseline):
    if np.isnan(baseline) or baseline == 0:
        return epoch - baseline
    return (epoch - baseline) / baseline * 100

def _sig(p):
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

def _bracket(ax, x1, x2, y_top, p, label):
    h = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.015
    ax.plot([x1, x1, x2, x2], [y_top, y_top+h, y_top+h, y_top], lw=1.2, color='k')
    ax.text((x1+x2)/2, y_top+h*1.5, f'{_sig(p)}  {label}  p={p:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

YLABEL = 'HR % change from baseline'

In [ ]:
# ── Session timecourse — full trace with startle markers ─────────────────────
from scipy.ndimage import gaussian_filter1d

def _clean_signal(sig, min_bpm, max_bpm):
    """Linear interpolation over artifact samples, then light smoothing."""
    s = sig.copy().astype(float)
    invalid = (s < min_bpm) | (s > max_bpm)
    if invalid.any():
        valid_idx = np.where(~invalid)[0]
        if len(valid_idx) > 1:
            s[invalid] = np.interp(np.where(invalid)[0], valid_idx, s[valid_idx])
    # Gaussian smooth: sigma=1s to soften the hardware BPM staircase
    sfreq_local = 1000.0
    s = gaussian_filter1d(s, sigma=sfreq_local * 1.0)
    return s

for subj, sessions in sorted(trials_data.items()):
    fig, axes = plt.subplots(2, 1, figsize=(18, 6), sharex=False)
    fig.suptitle(f'{subj} — HR Session Timecourse',
                 fontsize=12, fontweight='bold')

    for ax, sess_key in zip(axes, ['eve', 'mor']):
        trace  = _cache.get('traces', {}).get(subj, {}).get(sess_key)
        trials = sessions.get(sess_key, [])
        if trace is None or not trials:
            ax.set_title(f'{sess_key.upper()} — no data'); ax.axis('off'); continue

        raw_sig  = trace['signal']
        sfreq    = trace['sfreq']
        signal   = _clean_signal(raw_sig, HR_MIN_BPM, HR_MAX_BPM)
        times    = np.arange(len(signal)) / sfreq
        startle_times = trace['startle_samps_sec']

        ax.plot(times, signal, linewidth=0.8, color='steelblue', alpha=0.9)

        for i, t_sec in enumerate(startle_times):
            if i < len(trials):
                t_obj = trials[i]
                clr = config.REJ_COLOR if t_obj['rejected'] else (
                      config.NEG_COLOR if t_obj['label'] == 1 else config.NEU_COLOR)
            else:
                clr = 'gray'
            ax.axvline(t_sec, color=clr, linewidth=1.0, alpha=0.6)

        ax.set_title(config.SESSION_MAP[sess_key]['label'], fontsize=11, fontweight='bold')
        ax.set_xlabel('Time in session (s)'); ax.set_ylabel('HR (BPM)')
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.legend(handles=[
            plt.Line2D([0],[0], color=config.NEG_COLOR, label='Negative'),
            plt.Line2D([0],[0], color=config.NEU_COLOR, label='Neutral'),
            plt.Line2D([0],[0], color=config.REJ_COLOR, label='Rejected'),
        ], fontsize=8, loc='upper right')
    plt.tight_layout(); plt.show()

In [ ]:
# ── Subject average timecourse (Eve | Mor) ────────────────────────────────────
for subj, sessions in sorted(trials_data.items()):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    fig.suptitle(f'{subj} — Average HR Timecourse', fontsize=12, fontweight='bold')

    for ax, sess_key in zip(axes, ['eve', 'mor']):
        trials = sessions.get(sess_key, [])
        neg_e, neu_e, times = [], [], None
        for t in trials:
            if t['rejected'] or t['epoch_anal'] is None: continue
            ep = _pct(t['epoch_anal'], t['baseline_mean'])
            (neg_e if t['label'] == 1 else neu_e).append(ep)
            if times is None: times = t['times_anal']
        if times is None:
            ax.set_title(f"{config.SESSION_MAP[sess_key]['label']} — no data"); continue
        for eps, clr, lbl in [(neg_e, config.NEG_COLOR, 'Negative'),
                               (neu_e, config.NEU_COLOR, 'Neutral')]:
            if not eps: continue
            arr = np.array(eps)
            avg = np.mean(arr, axis=0)
            sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr)>1 else np.zeros_like(avg)
            ax.plot(times, avg, color=clr, linewidth=2.2, label=f'{lbl} (n={len(eps)})')
            ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
        ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
        ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
        ax.axvspan(config.SCORE_TMIN, config.SCORE_TMAX, color='green', alpha=0.07)
        ax.axvspan(config.BASELINE_TMIN, config.BASELINE_TMAX, color='gold', alpha=0.07)
        ax.set_title(config.SESSION_MAP[sess_key]['label'], fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (s)'); ax.set_ylabel(YLABEL); ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Group average timecourse ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle('Group Average HR Timecourse', fontsize=13, fontweight='bold')

for ax, sess_key in zip(axes, ['eve', 'mor']):
    neg_e, neu_e, times = [], [], None
    n_neg_s = n_neu_s = 0
    for subj, sessions in trials_data.items():
        trials = sessions.get(sess_key, [])
        s_neg = [_pct(t['epoch_anal'], t['baseline_mean'])
                 for t in trials if not t['rejected'] and t['label']==1
                 and t['epoch_anal'] is not None]
        s_neu = [_pct(t['epoch_anal'], t['baseline_mean'])
                 for t in trials if not t['rejected'] and t['label']==2
                 and t['epoch_anal'] is not None]
        if s_neg: neg_e.append(np.mean(s_neg, axis=0)); n_neg_s += 1
        if s_neu: neu_e.append(np.mean(s_neu, axis=0)); n_neu_s += 1
        if times is None:
            times = next((t['times_anal'] for t in trials if t['times_anal'] is not None), None)
    if times is None:
        ax.set_title(f"{config.SESSION_MAP[sess_key]['label']} — no data"); continue
    for eps, clr, lbl, ns in [(neg_e, config.NEG_COLOR, 'Negative', n_neg_s),
                               (neu_e, config.NEU_COLOR, 'Neutral',  n_neu_s)]:
        if not eps: continue
        arr = np.array(eps)
        avg = np.mean(arr, axis=0)
        sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr)>1 else np.zeros_like(avg)
        ax.plot(times, avg, color=clr, linewidth=2.2, label=f'{lbl} (n subj={ns})')
        ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
    ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvspan(config.SCORE_TMIN, config.SCORE_TMAX, color='green', alpha=0.07)
    ax.axvspan(config.BASELINE_TMIN, config.BASELINE_TMAX, color='gold', alpha=0.07)
    ax.set_title(config.SESSION_MAP[sess_key]['label'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (s)'); ax.set_ylabel(YLABEL); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── Overall timecourse: Evening vs Morning ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
times = None
for sess_key, clr, lbl in [('eve', config.EVE_COLOR, 'Evening'),
                             ('mor', config.MOR_COLOR, 'Morning')]:
    subj_avgs = []
    for subj, sessions in trials_data.items():
        trials = sessions.get(sess_key, [])
        accepted = [t for t in trials if not t['rejected'] and t['epoch_anal'] is not None]
        if not accepted: continue
        subj_avgs.append(np.mean([_pct(t['epoch_anal'], t['baseline_mean'])
                                   for t in accepted], axis=0))
        if times is None: times = accepted[0]['times_anal']
    if not subj_avgs or times is None: continue
    arr = np.array(subj_avgs)
    avg = np.mean(arr, axis=0)
    sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr)>1 else np.zeros_like(avg)
    ax.plot(times, avg, color=clr, linewidth=2.5, label=f'{lbl} (n={len(subj_avgs)})')
    ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.axvspan(config.SCORE_TMIN, config.SCORE_TMAX, color='green', alpha=0.07)
ax.axvspan(config.BASELINE_TMIN, config.BASELINE_TMAX, color='gold', alpha=0.07)
ax.set_xlabel('Time (s)', fontsize=11); ax.set_ylabel(YLABEL, fontsize=11)
ax.set_title('Overall Average HR: Evening vs Morning (±SEM)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── Group score boxplot (4 conditions) ───────────────────────────────────────
cond_keys = [('eve', 1, 'Eve-Neg', config.NEG_COLOR),
             ('eve', 2, 'Eve-Neu', config.NEU_COLOR),
             ('mor', 1, 'Mor-Neg', '#922B21'),
             ('mor', 2, 'Mor-Neu', '#1A5276')]

subj_means = {}
for subj, sessions in trials_data.items():
    subj_means[subj] = {}
    for sk, cond, *_ in cond_keys:
        vals = [t['score'] for t in sessions.get(sk, [])
                if not t['rejected'] and t['label'] == cond
                and t['score'] is not None and not np.isnan(t['score'])]
        if vals:
            subj_means[subj][(sk, cond)] = np.mean(vals)

valid = [s for s in subj_means if all((sk,c) in subj_means[s] for sk,c,*_ in cond_keys)]
if len(valid) < 2:
    print('Not enough subjects')
else:
    data_cols = [[subj_means[s][(sk,c)] for s in valid] for sk,c,*_ in cond_keys]
    n_trials  = [sum(1 for s in valid for t in trials_data[s].get(sk,[])
                     if not t['rejected'] and t['label']==c) for sk,c,*_ in cond_keys]
    fig, ax = plt.subplots(figsize=(10, 6))
    bp = ax.boxplot(data_cols, positions=[1,2,3,4], widths=0.45, patch_artist=True, showfliers=False)
    for patch,(sk,c,lbl,clr) in zip(bp['boxes'], cond_keys):
        patch.set_facecolor(clr); patch.set_alpha(0.65); patch.set_edgecolor('gray')
    for med in bp['medians']: med.set(color='black', linewidth=2)
    for s in valid:
        j = np.random.uniform(-0.07, 0.07)
        for (pn,pu),sk in [((1,2),'eve'),((3,4),'mor')]:
            vn = subj_means[s].get((sk,1)); vu = subj_means[s].get((sk,2))
            if vn is not None and vu is not None:
                lc = config.UP_COLOR if vu>=vn else config.DOWN_COLOR
                ax.plot([pn+j,pu+j],[vn,vu],color=lc,alpha=0.45,lw=1.2)
                ax.scatter(pn+j,vn,color=cond_keys[0 if pn==1 else 2][3],edgecolors='k',s=45,zorder=4,lw=0.5)
                ax.scatter(pu+j,vu,color=cond_keys[1 if pu==2 else 3][3],edgecolors='k',s=45,zorder=4,lw=0.5)
    for ca,cb,x1,x2 in [(data_cols[0],data_cols[1],1,2),(data_cols[2],data_cols[3],3,4)]:
        try:    _,p = stats.wilcoxon(ca,cb); tl='Wilcoxon'
        except: _,p = stats.ttest_rel(ca,cb); tl='Paired t'
        y_top = max(max(ca),max(cb))*1.08
        ax.set_ylim(top=max(ax.get_ylim()[1], y_top*1.25))
        _bracket(ax,x1,x2,y_top,p,tl)
    ax.set_xticks([1,2,3,4])
    ax.set_xticklabels([f'{lbl}\n(n={nt})' for (_,_,lbl,_),nt in zip(cond_keys,n_trials)])
    ax.set_ylabel(YLABEL, fontsize=11)
    ax.set_title(f'HR Score by Condition  (N={len(valid)} subjects)', fontweight='bold')
    ax.axvline(2.5, color='lightgray', linestyle='--', lw=1)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Eve vs Mor — Subjective Negative, sound-included trials only ──────────────
# Score = (max HR in 0-6s post-startle - mean baseline -5-0s) / baseline * 100
eve_scores, mor_scores, valid_subjs = [], [], []

for subj, sessions in sorted(trials_data.items()):
    s = {}
    for sk in ['eve', 'mor']:
        vals = [t['score'] for t in sessions.get(sk, [])
                if not t['rejected']
                and t.get('subjective_label', t['label']) == 1
                and t['score'] is not None and not np.isnan(t['score'])]
        if vals: s[sk] = np.mean(vals)
    if 'eve' in s and 'mor' in s:
        eve_scores.append(s['eve']); mor_scores.append(s['mor']); valid_subjs.append(subj)

n = len(valid_subjs)
if n < 2:
    print('Not enough subjects')
else:
    try:    _, p = stats.wilcoxon(eve_scores, mor_scores); tl = 'Wilcoxon'
    except: _, p = stats.ttest_rel(eve_scores, mor_scores); tl = 'Paired t'
    fig, ax = plt.subplots(figsize=(6, 6))
    bp = ax.boxplot([eve_scores, mor_scores], positions=[1,2], widths=0.4,
                    patch_artist=True, showfliers=False)
    for patch, clr in zip(bp['boxes'], [config.EVE_COLOR, config.MOR_COLOR]):
        patch.set_facecolor(clr); patch.set_alpha(0.6); patch.set_edgecolor('gray')
    for med in bp['medians']: med.set(color='black', linewidth=2)
    for ev, mo in zip(eve_scores, mor_scores):
        j = np.random.uniform(-0.06, 0.06)
        lc = config.UP_COLOR if mo >= ev else config.DOWN_COLOR
        ax.plot([1+j,2+j],[ev,mo],color=lc,alpha=0.55,linewidth=1.2)
        ax.scatter(1+j,ev,color=config.EVE_COLOR,edgecolors='k',s=55,zorder=4,linewidths=0.5)
        ax.scatter(2+j,mo,color=config.MOR_COLOR,edgecolors='k',s=55,zorder=4,linewidths=0.5)
    h = (ax.get_ylim()[1]-ax.get_ylim()[0])*0.015
    y_top = max(max(eve_scores),max(mor_scores))*1.08
    ax.set_ylim(top=max(ax.get_ylim()[1], y_top*1.25))
    ax.plot([1,1,2,2],[y_top,y_top+h,y_top+h,y_top],lw=1.2,color='k')
    ax.text(1.5,y_top+h*1.5,f'{_sig(p)}  {tl}  p={p:.3f}',
            ha='center',va='bottom',fontsize=9,fontweight='bold')
    ax.set_xticks([1,2])
    ax.set_xticklabels([f'Evening\n(n={n})',f'Morning\n(n={n})'],fontsize=11)
    ax.set_ylabel(YLABEL, fontsize=11)
    ax.set_title(f'HR Score — Subjective Negative\nEvening vs Morning  (N={n} subjects)',
                 fontweight='bold')
    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    plt.tight_layout(); plt.show()
    print(f'Eve mean={np.mean(eve_scores):.2f}%  Mor mean={np.mean(mor_scores):.2f}%  p={p:.3f}')